In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

print('Imports OK')

In [ ]:
df = pd.read_csv('../data/processed_dataset.csv')
print(f'Dataset chargé : {df.shape}')
df.head()

In [ ]:
features = ['hr', 'cad', 'alt', 'vitesse_kmh', 'pente_pct',
            'acceleration', 'delta_hr', 'vitesse_moy_5s',
            'hr_moy_5s', 'cad_moy_5s', 'pente_moy_5s']

cible = 'power'

print('Features :', features)
print('Cible :', cible)

In [ ]:
tous_les_rides = sorted(
    [r for r in df['ride_id'].unique() if str(r).isdigit()],
    key=lambda x: int(x)
)
df = df[df['ride_id'].isin(tous_les_rides)]

rides_test  = tous_les_rides[-20:]
rides_train = tous_les_rides[:-20]

df_train = df[df['ride_id'].isin(rides_train)]
df_test  = df[df['ride_id'].isin(rides_test)]

X_train = df_train[features]
y_train = df_train[cible]
X_test  = df_test[features]
y_test  = df_test[cible]

print(f'Train : {X_train.shape[0]} lignes')
print(f'Test  : {X_test.shape[0]} lignes')

In [ ]:
print('Régression linéaire...')
modele_lineaire = LinearRegression()
modele_lineaire.fit(X_train, y_train)

pred_lineaire = modele_lineaire.predict(X_test)
r2_lin   = r2_score(y_test, pred_lineaire)
mae_lin  = mean_absolute_error(y_test, pred_lineaire)
rmse_lin = np.sqrt(mean_squared_error(y_test, pred_lineaire))

print(f'R² : {r2_lin:.4f}  MAE : {mae_lin:.2f} W  RMSE : {rmse_lin:.2f} W')

In [ ]:
print('Random Forest...')
modele_rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
modele_rf.fit(X_train, y_train)

pred_rf   = modele_rf.predict(X_test)
r2_rf     = r2_score(y_test, pred_rf)
mae_rf    = mean_absolute_error(y_test, pred_rf)
rmse_rf   = np.sqrt(mean_squared_error(y_test, pred_rf))

print(f'R² : {r2_rf:.4f}  MAE : {mae_rf:.2f} W  RMSE : {rmse_rf:.2f} W')

In [ ]:
print('XGBoost...')
modele_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0)
modele_xgb.fit(X_train, y_train)

pred_xgb   = modele_xgb.predict(X_test)
r2_xgb     = r2_score(y_test, pred_xgb)
mae_xgb    = mean_absolute_error(y_test, pred_xgb)
rmse_xgb   = np.sqrt(mean_squared_error(y_test, pred_xgb))

print(f'R² : {r2_xgb:.4f}  MAE : {mae_xgb:.2f} W  RMSE : {rmse_xgb:.2f} W')

In [ ]:
resultats = pd.DataFrame({
    'Modèle':    ['Régression Linéaire', 'Random Forest', 'XGBoost'],
    'R²':        [r2_lin,   r2_rf,   r2_xgb],
    'MAE (W)':   [mae_lin,  mae_rf,  mae_xgb],
    'RMSE (W)':  [rmse_lin, rmse_rf, rmse_xgb]
})
print(resultats.round(4).to_string(index=False))

In [ ]:
echantillon = np.random.choice(len(y_test), size=3000, replace=False)
y_reel   = np.array(y_test)[echantillon]
y_predit = pred_xgb[echantillon]

plt.figure(figsize=(8, 8))
plt.scatter(y_reel, y_predit, alpha=0.2, s=5, color='steelblue')
plt.plot([0, 1000], [0, 1000], 'r--', linewidth=1, label='Prédiction parfaite')
plt.title(f'XGBoost : Réel vs Prédit (R²={r2_xgb:.3f})')
plt.xlabel('Puissance réelle (W)')
plt.ylabel('Puissance prédite (W)')
plt.legend()
plt.tight_layout()
plt.savefig('../plots/xgboost_reel_vs_predit.png')
plt.show()

In [ ]:
importances = pd.Series(modele_xgb.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Importance des features - XGBoost')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../plots/feature_importance.png')
plt.show()

In [ ]:
joblib.dump(modele_lineaire, '../models/linear_regression.joblib')
joblib.dump(modele_rf,       '../models/random_forest.joblib')
joblib.dump(modele_xgb,      '../models/xgboost.joblib')
print('Modèles sauvegardés')